# Three-Way Ablation: Reuters vs Style vs Both

Tests the full causal chain in one run:
1. **Original** — Reuters present, style artifacts present (should reproduce ~100%)
2. **No Reuters** — Reuters removed only (should reproduce ~99.9%, matches your last result)
3. **No Reuters + De-styled** — Reuters removed AND exclamation/question/ALL-CAPS patterns neutralized

If de-styling causes a real drop where Reuters-removal alone didn't, that confirms punctuation/capitalization -- not lexical leakage -- is the dominant driver.

**Before running:** Runtime -> T4 GPU. Upload the new `kaggle_clean.csv` (has the `transformer_text_no_reuters_destyled` column).

In [ ]:
!pip install -q transformers datasets torch scikit-learn pandas

## Imports and setup

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

print("GPU available:", torch.cuda.is_available())

MODEL_NAME = "distilbert-base-uncased"
LABEL_MAP = {"FAKE": 0, "REAL": 1}
MAX_LENGTH = 256

## Reusable training function

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, pos_label=0),
        "recall": recall_score(labels, preds, pos_label=0),
        "f1": f1_score(labels, preds, pos_label=0),
    }


def train_and_eval(text_column, run_name, n_samples=15000, epochs=3, batch_size=16):
    print(f"\n{'='*60}\nRun: {run_name}  (text column: {text_column})\n{'='*60}")

    df = pd.read_csv("kaggle_clean.csv")
    df = df.dropna(subset=[text_column, "binary_label"])
    df["label"] = df["binary_label"].map(LABEL_MAP)
    df = df[[text_column, "label"]].rename(columns={text_column: "text"})
    df = df.sample(n=n_samples, random_state=42).reset_index(drop=True)

    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])
    print(f"Train: {len(train_df)} | Test: {len(test_df)}")
    print("Class balance in test set:", test_df["label"].value_counts().to_dict())

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    def tokenize(batch):
        return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

    train_ds = Dataset.from_pandas(train_df.reset_index(drop=True)).map(tokenize, batched=True)
    test_ds = Dataset.from_pandas(test_df.reset_index(drop=True)).map(tokenize, batched=True)

    training_args = TrainingArguments(
        output_dir=f"./results_{run_name}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        logging_steps=50,
    )

    trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)

    start = time.time()
    trainer.train()
    elapsed_min = (time.time() - start) / 60

    metrics = trainer.evaluate()
    preds = trainer.predict(test_ds)
    pred_labels = np.argmax(preds.predictions, axis=-1)
    cm = confusion_matrix(test_ds["label"], pred_labels, labels=[0, 1])

    print(f"\nTraining time: {elapsed_min:.1f} min")
    print(f"Accuracy: {metrics['eval_accuracy']:.4f} | F1: {metrics['eval_f1']:.4f}")
    print(f"Confusion matrix [FAKE/REAL]:\n{cm}")

    return {
        "run": run_name, "accuracy": metrics["eval_accuracy"], "precision": metrics["eval_precision"],
        "recall": metrics["eval_recall"], "f1": metrics["eval_f1"], "training_time_min": round(elapsed_min, 1),
    }

## Run 1: Original (Reuters + style present)

In [ ]:
r1 = train_and_eval("transformer_text", "original")

## Run 2: No Reuters

In [ ]:
r2 = train_and_eval("transformer_text_no_reuters", "no_reuters")

## Run 3: No Reuters + De-styled

In [ ]:
r3 = train_and_eval("transformer_text_no_reuters_destyled", "no_reuters_destyled")

## Compare all three

In [ ]:
comparison = pd.DataFrame([r1, r2, r3])
comparison["accuracy_drop_from_original"] = comparison["accuracy"].iloc[0] - comparison["accuracy"]
comparison.to_csv("three_way_ablation_results.csv", index=False)
print(comparison.to_string(index=False))

drop_reuters = r1["accuracy"] - r2["accuracy"]
drop_style = r2["accuracy"] - r3["accuracy"]
print(f"\nDrop from removing Reuters alone: {drop_reuters*100:.2f} pts")
print(f"Additional drop from de-styling (on top of Reuters removal): {drop_style*100:.2f} pts")

if drop_style > drop_reuters and drop_style > 0.01:
    print("=> Style artifacts explain MORE than lexical leakage. Confirms structural-check hypothesis.")
elif drop_style < 0.005 and drop_reuters < 0.005:
    print("=> Neither intervention meaningfully changed accuracy. Model may be robust to these specific artifacts -- worth investigating further (embeddings/semantic-level shortcuts, not surface-level).")
else:
    print("=> Mixed result -- report both effect sizes honestly in the paper.")

## Download results

In [ ]:
from google.colab import files
files.download("three_way_ablation_results.csv")